### Importing Packages


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns
import re

## Data Loading and Inspection


### Observations

##### I have noticed that there are four properties that has 0 Bedrooms, but only one of them is a studio type, and the description for all of them mentions that there are bedrooms, so maybe I will fix that manually
##### I need to check the distribution of area, Bedrooms to check for outliers.
##### There are two properties has a sale price of 1 which does not make sense, and they seems to be an outliers


In [ ]:
# Loading the Dataset
df = pd.read_csv(r'../data/apartments.csv')

# Adding a unique identifier for each apartment 
df['apartment_id'] = np.arange(1,len(df)+1)

# Data overview and inspection
print(f'The first few rows of the Dataset:')
display(df.head())

print('Dataset Information:')
df.info()
print(f'The data frame has {df.shape[0]} rows, and {df.shape[1]} columns.')

print('Summary Statistics:')
display(df.describe(include=np.number))

print('Missing Values per column:')
display(df.isnull().sum().sort_values(ascending=False))

print("Duplicated Rows:")
display(df.duplicated().sum())



In [ ]:
# Displaying the apartments that have 0 bedrooms
print('Apartments with 0 bedrooms:\n')
display(df[df['Bedrooms'] == 0])
print('---'*20)

# Displaying the apartments that have the minimum sale price
print('Apartments with the lowest sale price are:\n')
display(df[df['Sale_price'] == df['Sale_price'].min()])
print('---'*20)

# Data Exploration

## Observations:

##### The DataFrame does not contain duplicated values

##### The len() function returns the number of rows in the Dataframe, in a column (pandas Series) including the NaN values.

##### The dtype of Bedrooms and Bathrooms columns is float so they need to be converted to int64 type, as it is not possible that a property has 3.5 bedrooms

##### There are 760 missing values in the Price_monthly column, and 621 missing value in the Price_annualy column, so we need to know exactly how many missing values are there in these columns since we have two listing types (sale,rent), so out of all the rent properties we need to get the total number of missing values of their prices

##### for the sale price column there is around 200 missing value, also here we need to get the missing values of the sale price out of the sale listing type only; because these results are not accurate since the dataset is a mixture of rent and sale so the missing values in the sale price column are not all a missing values for the sale_price column actually and they are missing because the property is not rent and the same thing for rent listing type  

In [ ]:
print(f'The Dataset contains {df.shape[0]} rows, and {df.shape[1]} columns.')

# Extracting the columns names for further analysis
columns = df.columns.to_list()
for col in columns:
    print(f'Information about the colunmn: {col}\n\
          Data type: {df[col].dtype}\n\
          Number of missing values in the column: {df[col].isnull().sum()}\n\
          Proportion of the missing values in the column: {df[col].isnull().sum()/len(df[col])*100:.2f}%\n')
    
# The len() function returns the number of rows in the Dataframe, in a column (pandas Series) 
# including the NaN values.

# Extracting the listing type distibution
print(f'The DataFrame contains {df['Listing_type'].value_counts().loc["sale"]} sale listings,'
      f'and {df['Listing_type'].value_counts().loc["rent"]} rent listings.')

plt.figure(figsize=(8,5))
bars = sns.countplot(data=df,
              x='Listing_type',
              palette='Set2')
for bar in bars.patches:
    y_value = bar.get_height()
    x_val = bar.get_x() + (bar.get_width()/2)
    if y_value > 0:
        plt.text(x_val,y_value,y_value,ha='center',va='bottom')
    bar.set_edgecolor('white')
    bar.set_hatch('/')
sns.despine()
plt.grid(axis='y',
         linestyle='--',
         linewidth=0.7,
         color='gray')
plt.gca().set_axisbelow(True)
plt.title('Distribution of Property types')
plt.tight_layout()
plt.show()
# subset the dataframe to include only the sale listings
print(f'The number of missing values in sale price for sale listings: '
      f'{df[df['Listing_type'] == 'sale']['Sale_price'].isnull().sum()}'
      )

# subset the dataframe to include only the sale listings
print(f'The number of missing values in monthly, and annualy price for rent listings:\n'
      f'{df[df['Listing_type'] == 'rent'][['Price_monthly','Price_annualy']].isnull().sum()}')



## Missing Values Analysis

### Observations

##### after subsetting the data we found that there was only 19 missing values in the sale price column, and 198 in the price_monthly, and 59 in the price_annualy.

##### now I need to solve the issue of the price for rent listings since there are two columns for the price(monthly,annually) so I need to choose one of them 
##### since the price_monthly column has 198 missing values out of 208 values, maybe I will drop the column, or I would impute the missing values in the Price_annualy column and calculate the monthly price from it

##### Note: when imputing and filling the prices columns, I should be aware that it is not correct to impute all the data, and subset for the appropriate listing type before imputation and then proceed with the imputation

In [ ]:
print(f'The DataFrame contains {df['Listing_type'].value_counts().loc["sale"]}'
       f' sale listings, and {df['Listing_type'].value_counts().loc["rent"]} rent listings.')

# subset the dataframe to include only the sale listings
print(f'The number of missing values in sale price for sale listings:'
       f' {df[df['Listing_type'] == 'sale']['Sale_price'].isnull().sum()}')

# subset the dataframe to include only the sale listings
print(f'The number of missing values in monthly, and annualy price for rent listings:\n'
      f'{df[df['Listing_type'] == 'rent'][['Price_monthly','Price_annualy']].isnull().sum()}')

### Observations

##### The DataFrame contains six different types of floor types
##### The location column has 36 different locations for the apartments, and the location is written in Arabic in the form of 'country,city,town'
##### The apartments in the DataFrame are distributed among 14 floors 
##### The description and the specialities features are going to be a big challenge as they are paragraphs 

In [ ]:
# Separating the categorical and numerical features for analysis

categorical_cols = df.select_dtypes(exclude=np.number).columns.to_list()
numerical_cols = df.select_dtypes(include=np.number).columns.to_list()

print(f'The categorical Columns are:\n{categorical_cols}\n')
print(f'The numerical Columns are:\n{numerical_cols}\n')

# Extracting the unique values for each column
print('The Unique values in some of categorical columns:')
display(df['Floor_type'].unique())
display(df['Listing_type'].unique())
display(df['Furnished'].nunique())
display(df['Location'].head())
display(df['Floor'].unique())
display(df['Location'].unique())


In [ ]:
display(df['Location'].value_counts().sort_values(ascending=False))
display(df['Specialities'].head(20))

'''df['Location'].iloc[10].strip(' ').split(',')
for row in df['Location']:
    if isinstance(row,str):
        location_parts = row.strip(' ').split(',')
    print(location_parts[0])
    df['City'] = location_parts[1]
    df['Town'] = location_parts[2]
df.head()
'''

df['Description'].head()

### Observations:

##### After reviewing the distributions of the area and the number of bedrooms, there is no outliers in the data; due to there are more than one floor type in the properties and 

In [ ]:
plt.figure(figsize=(10,6),dpi=100)
# Plotting the distribution of the area for the apartments
sns.histplot(data=df,
             x='Area_sqm',
             bins=30,
             kde=True,
             color='skyblue',
             edgecolor='white',
             hue='Listing_type')
sns.despine(right=True,top=True,left=False)
plt.grid(axis='y',linestyle='--',linewidth=0.7,color='gray')
plt.gca().set_axisbelow(True)
plt.title('Distribution of the area of the apartments')
plt.xticks(ticks=np.arange(0,df['Area_sqm'].max()+50,50))
plt.tight_layout()
plt.show()

# Distribution of the number of Bedrooms
plt.figure(figsize=(10,6),dpi=100)
sns.histplot(data=df,
             x='Bedrooms',
             bins=30,
             kde=True,
             color='salmon',
             edgecolor='white',
             hue='Listing_type')
sns.despine()
plt.grid(axis='y',linestyle='--',linewidth=0.7,color='gray')
plt.gca().set_axisbelow(True)
plt.title('Distribution of the number of Bedrooms')
plt.tight_layout()
plt.show()


In [ ]:
df[df['Area_sqm'].isin([df['Area_sqm'].min(), df['Area_sqm'].max()])]
df['Floor_type'].value_counts().sort_values(ascending=False)

## Data Cleaning Plan

##### - Drop unnecessary columns such as URL column
##### - Convert Bedrooms, and Bathrooms columns to Int64 data type instead of float
##### - Handle missing values for the price columns(Price_monthly,Price_annually,Sale_price)
##### - Since the Price_monthly column contains 198 missing values, I will impute the missing values in the Price_annually column and compute the Price_monthly from it 
##### - The data contains two observations with sale price of 1 which does not make sense, so I will drop those observations, I will impute the values based on the mean value of properties similar to them
##### - The specialities feature contains 53 missing values, and the description column contains 35 missing values and these values will be replaced by 'Unkown'
##### - for the most of the missing values in the categorical features they will be imputed by the most frequent value in the column
##### - for the missing values in the Bathrooms column, I will fill the missing values with the median of Bathrooms for each group of bedrooms, or I can extract the number from the specialities or description columns, as well as for the Bedrooms, Floor_type, Bedrooms, Location, area_sqm and Floor features but I will left it for an advanced stage

In [ ]:
df['Sale_price'].min()
df[df['Sale_price'] == df['Sale_price'].min()]

df.groupby('Bedrooms')['Bathrooms'].median()

### Experimental Feature recovery:
##### I want to test if I can extract the missing values for some features from the specialities and the description columns if applicable, so I want to examine if those values are available for the missing values of rows from the Bathrooms, Bedrooms, and prices columns

In [ ]:
# subsetting the DataFrame to include only the observations where the Bathrooms is null
print('Apartments with missing values in the Bathrooms column:\n')
display(df[df['Bathrooms'].isna()][['Specialities','Description']])

# subsetting the DataFrame to include only the observations where the Bedrooms is null
print('Apartments with missing values in the Bedrooms column:\n')
display(df[df['Bedrooms'].isna()][['Specialities','Description']])

# subsetting the DataFram to include only the observations where the Price_annually is null
print('Apartments with missing values in the Price_annualy column:\n')
display(df[(df['Price_annualy'].isna()) & (df['Listing_type'] == 'rent')][['Specialities','Description']])

# subsetting the DataFrame to only include the observations where the Sale_price is null
print('Apartments with missing values in the Sale_price column:\n')
display(df[(df['Sale_price'].isna()) & (df['Listing_type'] == 'sale')][['Specialities','Description']])

In [ ]:
before = df['Bathrooms'].isna().sum()

# I want to write a function to extract the number of bathrooms from the 
# Description and Specialities columns using regex

def extract_bathrooms_num(text):
    # if the text is null, then we cant search for the number of bathrooms and it will be Null
    if pd.isna(text):
        return None
    
    # check for arabic pattern
    # I will not check for the english pattern because the data is only in Arabic
    match_ar = re.search(r'(\d+)\s*حمامات',text)
    match_ar2 = re.search(r'حمامات\s*عدد\s*(\d+)',text)
    match_ar3 = re.search(r'(\d+)\s*حمام',text)
    
    if match_ar:
        return int(match_ar.group(1))
    elif match_ar2:
        return int(match_ar2.group(1))
    elif match_ar3:
        return int(match_ar3.group(1))
    
    # if no match were found return None
    return None
     
# now I want to write a function to fill the missing values in the Bathrooms column
# I will apply the extract_bathrooms_num function to both the Description and Specialities columns
# But I need to that I will only fill the missing values in the Bathrooms column, not the whole column

def fill_bathrooms_num(row):
    # if the value is not missing, then we will keep it as it is
    if pd.notna(row['Bathrooms']):
        return row['Bathrooms']
    value = extract_bathrooms_num(row['Description'])
    # if the regex didnt find any match in the description column (the regex will return None), then I will try to extract 
    # the number from the Specialities column
    if value is None:
        value = extract_bathrooms_num(row['Specialities'])
    return value

# now I will apply the fill_bathrooms_num function to the DataFrame where the number 
# of bathrooms is null

df.loc[df['Bathrooms'].isna(),'Bathrooms'] = df.loc[df['Bathrooms'].isna(),:].apply(fill_bathrooms_num,axis=1)

print(f'Filled {before - df["Bathrooms"].isna().sum()} missing values from the Bathrooms column')
print(f'Missing before {before}')
print(f'Missing after {df["Bathrooms"].isna().sum()}')


In [ ]:
before = df[df['Listing_type'] == 'rent']['Price_annualy'].isna().sum()

# Annualy price for rent listing types extraction 
def extract_annualy_price(text):
    if pd.isna(text):
        return None
    
    # we will check for arabic pattern since the DataFrame is in arabic
    patterns = [
        r"السعر\s*\(?سنوي(?:ا|اً)?\)?\s*[:\-]?\s*([\d,]+)",
        r"سنوي(?:ا|اً)?\s*[:\-]?\s*([\d,]+)",
        r"([\d,]+)\s*دينار(?:\s*اردني)?\s*(?:سنوي|سنويا|سنوياً)",
        r"السعر\s*[:\-]?\s*([\d,]+)",
    ]
    for pattern in patterns:
        match_ar = re.search(pattern, text)
        if match_ar:
            return int(match_ar.group(1).replace(',',''))
    return None

def fill_annually_price(row):
    if pd.notna(row['Price_annualy']):
        return row['Price_annualy']
    
    # Trying to extract the value from the specialities column first
    value = extract_annualy_price(row['Specialities'])
    if value is None:
        value = extract_annualy_price(row['Description'])
    return value

#df.loc[df['Price_annualy'].isna(),'Price_annualy'] = df.loc[df['Price_annualy'].isna(),:]\
#.apply(fill_annually_price,axis=1)

# making sure to only fill the missing values of the Price_annualy column for the rent listing type
rent_mask = (df['Listing_type'] == 'rent') & (df['Price_annualy'].isna())
#df[df['Listing_type'] == 'rent'].loc[df['Price_annualy'].isna(),'Price_annualy'] =\
#df[df['Listing_type'] == 'rent'].loc[df['Price_annualy'].isna(),:].apply(fill_annually_price,axis=1)
df.loc[rent_mask,'Price_annualy'] = df.loc[rent_mask,:].apply(fill_annually_price,axis=1)


print(f'Filled {before - df[df["Listing_type"] == "rent"]["Price_annualy"].isna().sum()} values')
print(f'Missing before {before}')
print(f'Missing after {df[df["Listing_type"] == "rent"]["Price_annualy"].isna().sum()}')



        

In [ ]:
# Checking for the number of invalid prices in the sale and rent listing types
display(df[(df['Listing_type'] == 'sale') & (df['Sale_price'] <= 1000)].shape[0])
display(df[(df['Listing_type'] == 'rent') & (df['Price_annualy'] <= 1000)].shape[0])

In [ ]:
# Checking for the details of the apartments that have invalid prices 
display(df[df['Sale_price'] <= 1000][['Description','Specialities']])
display(df.loc[df['Sale_price'] <= 1000,['Sale_price','Specialities']])


##### There is an issue in 25 rows in the sale price column for sale listing type, that the prices are invalid and they are less than 100 so we need to try to extract the prices for these invalid observations from the description or the specialities column 

In [ ]:
# Now I am going to try to extract the sale price from the description or specialities columns
# for the missing values of the Sale_price column and the invalid values that are less than 1000
#  for the sale listing type

def extract_sale_price(text):
    if pd.isna(text):
        return None
    
    # we will only be checking for an arabic pattern since the data is only in arabic
    patterns = [
        r"السعر\s*[:\-]?\s*([\d,]+)",
        r"سعر\s*\(?البيع\)?\s*[:\-]?\s*([\d,]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern=pattern,string=text)
        if match:
            return int(match.group(1).replace(',',''))
    return None

def fill_sale_price(row): # function for filling the missing values in the sale price column, where the listing type is sale
    if pd.notna(row['Sale_price']):
        return row['Sale_price']
    
    value = extract_sale_price(row['Description'])
    if value is None:
        value = extract_sale_price(row['Specialities'])
    return value

before_missing = df[df['Listing_type'] == 'sale']['Sale_price'].isna().sum() 
before_invalid = df[(df['Listing_type'] == 'sale') & (df['Sale_price'] <= 1000)]['Sale_price'].count
# missing values before filling
# making sure to only fill the missing values of the Sale_price column for& the sale listing type
sale_mask = (df['Listing_type'] == 'sale') & (df['Sale_price'].isna())
df.loc[sale_mask,'Sale_price'] = df.loc[sale_mask,:].apply(fill_sale_price,axis=1)

print(f'Filled {before - (df[df['Listing_type'] == 'sale']['Sale_price'].isna().sum())}' 
      f' missing values from the Sale_price column')
print(f'Missing values in Sale_price column before filling: {before}')
print(f'Missing values in the Sale_price column after filling:'
      f'{df[df['Listing_type'] == 'sale']['Sale_price'].isna().sum()}')


In [ ]:

def extract_price(text):
    if pd.isna(text):
        return None
    
    patterns = [
        r"السعر\s*[:\-]?\s*([\d,]+)",
        r"سعر\s*\(?البيع\)?\s*[:\-]?\s*([\d,]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern=pattern,string=text)
        if match:
            return int(match.group(1).replace(',',''))
    return None

def fill_price(row): 
    # function for filling the missing values in the sale price column, where the listing type is sale
    if (pd.notna(row['Sale_price'])) and (row['Sale_price'] > 1000):
        return row['Sale_price']
    
    value = extract_price(row['Description'])
    if value is None:
        value = extract_price(row['Specialities'])
    return  row['Sale_price'] if value is None else value

before = df[(df['Listing_type'] == 'sale') & (df['Sale_price'] <= 1000)]
mask = (df['Listing_type'] == 'sale') & (df['Sale_price'] <= 1000)
df.loc[mask,'Sale_price'] = df.loc[mask,:].apply(fill_price,axis=1)
print(f'extracted price for {before.shape[0] - (df[(df['Listing_type'] == 'sale') & (df['Sale_price'] <= 1000)].shape[0])} ')
print(f'The invalid values in the sale price column before filling: {before.shape[0]}')
print(f'The invalid values in the sale price column after filling: {df[(df['Listing_type'] == 'sale') & (df['Sale_price'] <= 1000)].shape[0]}')

In [ ]:
display(df[df['Bedrooms'] == 0][['Description']])
print(f'The number of invalid values in the Bedrooms column is:'
      f'{df[df['Bedrooms'] == 0].shape[0]}')
print(f'the number of missing values in the Bedrooms column is:'
      f'{df['Bedrooms'].isna().sum()}' )


##### Observations
##### The DataFrame contains four rows with Bedrooms value with zero, so we need to extract the number of bedrooms for those rows from the description or the specialities columns

#### Note I need to fix the extracting bedrooms number function logic

In [ ]:
# Extracting the no. of bedrooms
before = df[(df['Bedrooms'] == 0) & (df['Bedrooms'].isna())].shape[0]

def extract_bedrooms_num(text):
    if pd.isna(text):
        return None

    patterns = [
        r'(\d+)\s*غرف\s*نوم',
        r'غرف\s*نوم\s*عدد\s*(\d+)',
        r'(\d+)\s*غرفة\s*نوم',
        r'غرفة\s*نوم\s*عدد\s*(\d+)'
    ]

    
    if pd.notna(text) and re.search(r"غرفة\s*نوم(?:\s*ماستر)?", str(text)):
        return 1

    if pd.notna(text) and re.search(r"غرف\s*نوم\s*ماستر", str(text)):
        return 1

    for pattern in patterns:
        match = re.search(pattern=pattern,string=str(text))
        
        if match:
            return int(match.group(1))
    return None

def fill_bedrooms_num(row):
    if row['Bedrooms'] !=0 and pd.notna(row['Bedrooms']):
        return row['Bedrooms']

    value = extract_bedrooms_num(row['Description'])
    if value is None:
        value = extract_bedrooms_num(row['Specialities'])
    return value if value is not None else np.nan

mask = (df['Bedrooms'] == 0)
df.loc[mask,'Bedrooms'] = df.loc[mask,:].apply(fill_bedrooms_num,axis=1)

print(f'Filled {before - (df[(df['Bedrooms'] == 0) & (df['Bedrooms'].isna())]).shape[0]} missing and invalid values')
print(f'Missing and invalid values before filling: {before}')
print(f'Missing and invalid values after filling: {(df[(df['Bedrooms'] == 0) & (df['Bedrooms'].isna())]).shape[0]}')


display(df[df['Bedrooms'] == 0])
df[df['Bedrooms'].isna()]

##### Observations from feature extraction 
##### for the Bathrooms missing values, there were 11 values extracted and the remaining missing values in the Bathrooms column is 7
##### for the Price_annualy column, there were 21 values extracted and the remaining missing values in the Price_annualy column for rent listing types are 38 values
##### for the Sale_price column, there were 4 values extracted and the remaining missing values in the Sale_price column for the sale listing type are 15 values
##### out of 25 invalid prices in the Sale_price column for the sale listing type, 2 values were extracted, and the unrecovered values will be turned into NaNs 


### Observations and Notes
##### some of the values missing values from the Batrooms, annualy_price, and Sale_price columns have been recovered and the number of missing values in these columns has been reduced, so:
#### 1- I wiil apply the feature recovery approach before handling the missing values
#### 2- The missing values will be handled as follows:
#####  - The missing values in the bathrooms column will be filled based on the median bathrooms for bedrooms number
#####  - Missing values in the Bedrooms column will be filled with the median of Bedrooms
#####  - Missing values in the Area_sqm column will be filled with the median area
#####  - Missing values in the floor and floor type columns will be filled with unknown 
#####  - Missing values in the description and specialities will be filled with whitespaces
#####  - Missing values in the sale price and the price annualy columns will be dropped  


In [ ]:
df.isna().sum().sort_values(ascending=False)

for col in df.columns:
    if col not in ['Price_monthly','Price_annualy','Sale_price']:
        print(f'Missing values in the {col} column: is {df[col].isna().sum()}')

display('The missing values in the Sale_price column for sale listings are:',df[df['Listing_type'] == 'sale']['Sale_price'].isna().sum())
display('The missing values in the Price_annualy column for rent listings are:',df[df['Listing_type'] == 'rent']['Price_annualy'].isna().sum())

In [ ]:
df[(df['Listing_type'] == 'sale') & (df['Sale_price'].isna())][['Description','Specialities']]

In [ ]:
df['Bathrooms'].isna().sum()
df.groupby('Bedrooms')['Bathrooms'].median()
df.groupby('Bedrooms')['Bathrooms'].transform(lambda x: x.fillna(x.median()))
display(df.columns)

In [ ]:
df[['Description','Specialities']] = df[['Description','Specialities']].fillna('')
df[['Floor_type','Floor']] = df[['Floor_type','Floor']].fillna('Unknown')
df['Area_sqm'] = df['Area_sqm'].fillna(df['Area_sqm'].median())
df['final_price'] = np.where(df['Listing_type'] == 'sale',df['Sale_price'],df['Price_annualy'])
df.dropna(subset=['final_price','Location'],inplace=True,axis=0)
df.drop(columns=['Price_monthly','Price_annualy','Sale_price'],inplace=True,errors='ignore')

df['Bathrooms'].astype(dtype='Int64')
df['Bedrooms'].astype(dtype='Int64')
df['Area_sqm'].astype(dtype='float64')
df['final_price'].astype(dtype='float64')




In [ ]:
df.shape
df.isna().sum()

In [ ]:
#df[df['Price_annualy'] < 1000]

### Feature engineering phase

##### for the floor column, the floors are mentioned as an objects, so I need to use mapping to them to the appropriate floor number
##### I can extract the city and the town from the location column using the string methods and apply it to the whole column
##### Feature extraction from the specialities column


In [ ]:
display(df.head())
# Now I need to create a new columns from the location column that contains the city and the area

display(df['Location'].head())

display(df.iloc[5]['Location'].strip(' ').split(',')[1])
df['Area'] = pd.NA
df['City'] = pd.NA

# Extracting the area and the city column by iterating over the location column
'''
for idx,val in df['Location'].items():
    if pd.isna(val):
        continue
    location_parts = [p.strip() for p in (val).split(',')]
    df.at[idx,'Area'] = location_parts[0] if len(location_parts) > 0 else pd.NA
    df.at[idx,'City'] = location_parts[1] if len(location_parts) > 1 else pd.NA
'''
# Using vectorization to extract area and city faster

df[['Area','City','Country']] = df['Location'].str.split(',',expand=True)
df['Area'] = df['Area'].str.strip()
df['City'] = df['City'].str.strip()
df['Country'] = df['Country'].str.strip()

display(df['Country'].unique())
display(df['City'].unique())
display(df['Area'].unique())

In [ ]:
df['price_per_sqm'] = df['final_price'].div(df['Area_sqm'],fill_value=np.nan)
df.head()

In [ ]:
for i in range(len(df)):
    print(df.iloc[i]['text'])

#### Feature Extraction from description and specialities columns
##### I am going to extract the most common important binary features from the text 
##### I have extracted also the number of salons and the masters bedrooms  

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ''
    return re.sub(r'\s+',' ',str(text)).strip()

df['text'] = (df['Description'] + ' ' + df['Specialities']).apply(normalize_text)
df['has_elevator'] = df['text'].str.contains(r'مصعد').astype(int)
df['has_parking'] = df['text'].str.contains(r'كراج').astype(int)
df['has_storage'] = df['text'].str.contains(r'مخزن').astype(int)
df['has_balcony'] = df['text'].str.contains(r'بلكونة|بلكون|بلكونه',regex=True).astype(int)
df['has_garden'] = df['text'].str.contains(r'حديقة').astype(int)
df['has_gym'] = df['text'].str.contains(r'جيم|صالة\s*رياضية|نادي\s*رياضي',regex=True).astype(int)
df['has_living_room'] = df['text'].str.contains(r'غرفة\s*معيشة|غرفة\s*معيشه',regex=True).astype(int)
df['has_kitchen'] = df['text'].str.contains(r'مطبخ').astype(int)
df['has_laundry_room'] = df['text'].str.contains(r'غرفة غسيل|غرفة\s*غسيل|غسيل').astype(int)
df['has_heating'] = df['text'].str.contains( r'تدفئة|تدفئة\s*مركزية|تدفئة\s*تحت\s*البلاط',regex=True).astype(int)
df['has_AC'] = df['text'].str.contains(r'تكييف|تكييف\s*مركزي|وحدات\s*تكييف',regex=True).astype(int)
df['has_security_doors'] = df['text'].str.contains( r'ابواب\s*امان|أبواب\s*أمان|باب\s*امان|باب\s*أمان',regex=True).astype(int)






In [ ]:
plt.figure(figsize=(10,6),dpi=100)
bars = sns.countplot(data=df,
              x='has_balcony',
              palette='flare')
for bar in bars.patches:
    y_val = bar.get_height()
    x_pos = bar.get_x() + (bar.get_width()/2)
    if y_val > 0:
        plt.text(x_pos,y_val,y_val,ha='center',va='bottom')
sns.despine()
plt.grid(axis='y',linestyle='--',linewidth=0.7,color='gray')
plt.gca().set_axisbelow(True)
plt.title('Distribution of apartments with balcony')
plt.tight_layout()
plt.show()

In [ ]:
'''for i in range(len(df['text'])):
    display(df.iloc[i]['text'])'''


df['salon'] = pd.NA

def extract_salons(text):
    if pd.isna(text):
        return 0
    
    patterns = [
        r'(\d+)\s*صالون',
        r'صالون\s*عدد\s*(\d+)'

    ]

    for pattern in patterns:
        match = re.search(pattern,str(text))
        if match:
            return int(match.group(1))
    if re.search(r'\bصالون\b',str(text)):
        return 1
    return 0

def fill_salons(row):
    value = extract_salons(row['text'])
    return value if value is not None else 0

df['salon'] = df.apply(fill_salons,axis=1)

df.head()
df['salon'].value_counts()
df.head()

In [ ]:
def extract_master_bedroom(text):
    if pd.isna(text):
        return 0
    patterns = [
        r'ماستر\s*عدد\s*(\d+)',
        r'(\d+)\s*ماستر',
        r'\((\d+)\s*ماستر\)',
        r'\(ماستر\s*(\d+)\)',
        r'\(ماستر\s*عدد\s*(\d+)\)'
    ]

    for pattern in patterns:
        match = re.search(pattern=pattern,
                          string=str(text))
        if match:
            return int(match.group(1))
    if re.search(pattern=r'غرفة\s*نوم\s*ماستر|نوم\s*ماستر',string=str(text)):
        return 1
    return 0

def fill_master_bedroom(row):
    value = extract_master_bedroom(row['text'])
    return value if value is not None else 0
df['master_bedrooms'] = df.apply(fill_master_bedroom,axis=1)

df.head()
df['master_bedrooms'].value_counts()

##### I have found that there are 4 observations in the Location column in the dataframe are invalid and the city and the area are not mentioned, so maybe I will fill those observations with the most frequent values for each column

##### I need to check and decide what features I want to extract from the combination of the description and specialities column
##### I want the features that I want to extract to be binary, like holding the value 1 and 0
##### After extracting new features then I will start encoding and standardizing the categorical and numerical features respectively by choosing the appropriate encoding and scaling methods


##### The most important binary features to extract would be parking, balcony, storage, elevator, garden, terrace


##### regarding the salons I did not extract it as a binary feature cause some properties comes with more than one salon